In [ ]:
"""
Temporal Fusion Transformer (Darts) — Flood Forecasting
Predicts streamflow 24 hours ahead for the top-30 high flood-severity sites.
Uses darts.models.TFTModel for built-in explainability (variable importance,
attention weights). Logs all metrics, charts, and artifacts to W&B.
"""
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import wandb

from darts import TimeSeries
from darts.models import TFTModel
from darts.explainability import TFTExplainer
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import EarlyStopping

from src.preprocessing.preprocessing import processor

In [ ]:
STATIC_FEATURES = [
    "longitude", "latitude", "DRAIN_SQKM", "artificial_path_pct",
    "wb5100_ann_mm", "snw_pc_syr", "snow_ice_nlcd06", "barren_nlcd06",
    "mains100_plant", "hga", "hgc", "bulk_density_avg", "elev_max_m", "aspect_deg",
]

DYNAMIC_FEATURES = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction",
]

# streamflow_cfs_mean is the TARGET; the remaining dynamic features are past covariates
PAST_COV_COLS = [c for c in DYNAMIC_FEATURES if c != "streamflow_cfs_mean"]

In [ ]:
wandb_config = {
    # Architecture
    "d_model": 64,
    "num_heads": 4,
    "hidden_continuous_size": 32,
    "lstm_layers": 1,
    "dropout_rate": 0.1,
    # Training
    "learning_rate": 1e-3,
    "under_predict_penalty": 2.0,
    "epochs": 5,
    "batch_size": 512,
    # Data
    "input_chunk_length": 72,
    "output_chunk_length": 24,
    "train_split": 0.8,
    "val_split": 0.9,
    "frequency": "hourly",
    "dataset": "flood-dataset-top30",
    "target": "streamflow_cfs_mean",
}

run = wandb.init(
    project="flood-forecasting",
    name="tft-darts",
    config=wandb_config,
    tags=["tft", "darts", "temporal-fusion-transformer", "top30"],
)

cfg = wandb.config
print(f"W&B run: {run.name}  |  id: {run.id}")

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
# lag_window=1 disables manual lag creation — Darts handles temporal context
# via input_chunk_length. No split_time_days so each site's series is continuous
# (required for Darts TimeSeries objects).
proc_config = {
    "input_cols": DYNAMIC_FEATURES + STATIC_FEATURES,
    "static_cols": STATIC_FEATURES,
    "target": "streamflow_cfs_target_24h",  # required by processor; not used by Darts
    "train_split": cfg.train_split,
    "val_split": cfg.val_split,
    "file_path": cfg.dataset,
    "file_name": "flood_model_top30",
    "table": "wandb.flood_model_top30",
    "lag_window": 1,
    "frequency": cfg.frequency,
    "site_scaling": False,
}

pcr = processor(proc_config)
pcr.pull_wandb()

# y outputs are unused — Darts infers the target shift from output_chunk_length
train_X, val_X, test_X, _, _, _ = pcr.return_outputs()

sites_ordered = sorted(train_X["site_id"].unique().to_list())
print(f"Sites: {len(sites_ordered)} | Train rows: {len(train_X):,} | Val: {len(val_X):,} | Test: {len(test_X):,}")

wandb.log({
    "data/n_sites": len(sites_ordered),
    "data/n_past_cov_features": len(PAST_COV_COLS),
    "data/n_static_features": len(STATIC_FEATURES),
})

In [ ]:
# ── Darts TimeSeries conversion ────────────────────────────────────────────────
# Per site: target = scaled streamflow_cfs_mean
#           past_covariates = all other scaled dynamic features
#           static_covariates attached to target TimeSeries

def to_darts(
    X_df: pl.DataFrame,
) -> tuple[list[TimeSeries], list[TimeSeries]]:
    """Convert a scaled Polars DataFrame to per-site Darts TimeSeries lists."""
    target_list: list[TimeSeries] = []
    past_cov_list: list[TimeSeries] = []

    for site in sorted(X_df["site_id"].unique().to_list()):
        site_pd = (
            X_df.filter(pl.col("site_id") == site)
            .sort("observation_hour")
            .to_pandas()
        )
        site_pd["observation_hour"] = pd.to_datetime(site_pd["observation_hour"])
        site_pd = site_pd.set_index("observation_hour").drop(columns=["site_id"])

        # Target: scaled streamflow_cfs_mean
        target_ts = TimeSeries.from_dataframe(
            site_pd[["streamflow_cfs_mean"]],
            fill_missing_dates=True,
            fillna_value=0.0,  # z-score mean=0 is a reasonable gap fill for scaled data
            freq="h",
        )

        # Attach static covariates (time-invariant — take first row)
        static_vals = pd.DataFrame(
            {col: [float(site_pd[col].iloc[0])] for col in STATIC_FEATURES}
        )
        target_ts = target_ts.with_static_covariates(static_vals)

        # Past covariates: all dynamic features except the target
        past_cov_ts = TimeSeries.from_dataframe(
            site_pd[PAST_COV_COLS],
            fill_missing_dates=True,
            fillna_value=0.0,
            freq="h",
        )

        target_list.append(target_ts)
        past_cov_list.append(past_cov_ts)

    return target_list, past_cov_list


print("Converting splits to Darts TimeSeries...")
train_series, train_past_covs = to_darts(train_X)
val_series,   val_past_covs   = to_darts(val_X)
test_series,  test_past_covs  = to_darts(test_X)
print(f"Done. Example train series length: {len(train_series[0])} steps")

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
TEST_RUN = False

class AsymmetricMSELoss(nn.Module):
    """MSE with configurable penalty on under-predictions (residual > 0)."""

    def __init__(self, under_penalty: float = 2.0):
        super().__init__()
        self.under_penalty = float(under_penalty)

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        error = target - pred
        weight = torch.where(error > 0, self.under_penalty, 1.0)
        return (weight * error.pow(2)).mean()


# Verify GPU
use_gpu = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if use_gpu else "CPU"
print(f"Device: {'GPU' if use_gpu else 'CPU'} — {device_name}")
print(f"CUDA version: {torch.version.cuda}")

from pytorch_lightning.callbacks import TQDMProgressBar

wandb_logger = WandbLogger(experiment=run, log_model=False)

tft = TFTModel(
    input_chunk_length=cfg.input_chunk_length,
    output_chunk_length=cfg.output_chunk_length,
    hidden_size=cfg.d_model,
    lstm_layers=cfg.lstm_layers,
    num_attention_heads=cfg.num_heads,
    dropout=cfg.dropout_rate,
    hidden_continuous_size=cfg.hidden_continuous_size,
    loss_fn=AsymmetricMSELoss(cfg.under_predict_penalty),
    optimizer_kwargs={"lr": cfg.learning_rate},
    batch_size=cfg.batch_size,
    use_static_covariates=True,
    add_relative_index=True,
    pl_trainer_kwargs={
        "max_epochs": cfg.epochs,
        "logger": wandb_logger,
        "callbacks": [
            TQDMProgressBar(refresh_rate=20),
        ],
        "accelerator": "gpu" if use_gpu else "cpu",
        "devices": 1,
        "enable_progress_bar": True,
        **({"fast_dev_run": 1} if TEST_RUN else {}),
    },
)

print(f"\nTFTModel ready | lookback: {cfg.input_chunk_length}h | horizon: {cfg.output_chunk_length}h")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
tft.fit(
    series=train_series,
    past_covariates=train_past_covs,
    val_series=val_series,
    val_past_covariates=val_past_covs,
    verbose=True,
)

total_params = sum(p.numel() for p in tft.model.parameters())
wandb.log({"model/total_params": total_params})
print(f"Training complete | Parameters: {total_params:,}")

In [ ]:
# ── Evaluation: rolling 24h-ahead forecasts over the test set ─────────────────
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

if TEST_RUN:
    # Smoke test: 1 site, just enough steps for a handful of predictions
    _n = cfg.input_chunk_length + cfg.output_chunk_length * 3
    eval_series    = [test_series[0][:_n]]
    eval_past_covs = [test_past_covs[0][:_n]]
    print(f"TEST_RUN: evaluating on 1 site, {_n} steps")
else:
    eval_series    = test_series
    eval_past_covs = test_past_covs

print("Running historical forecasts on test set...")
historical_preds = tft.historical_forecasts(
    series=eval_series,
    past_covariates=eval_past_covs,
    forecast_horizon=cfg.output_chunk_length,
    stride=cfg.output_chunk_length,
    start=cfg.input_chunk_length,
    retrain=False,
    last_points_only=True,
    enable_optimization=False,
    verbose=False,
)

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
print(f"Forecasts generated for {len(historical_preds)} series")

In [ ]:
# ── Postprocessing: inverse transform + global metrics ────────────────────────
pred_scaled_all:   list[float] = []
actual_scaled_all: list[float] = []
test_site_ids:     list[str]   = []

all_test_sites = sorted(test_X["site_id"].unique().to_list())
# In TEST_RUN only 1 site was evaluated; align site labels to eval_series length
eval_site_ids = all_test_sites[:len(historical_preds)]

for i, pred_ts in enumerate(historical_preds):
    site = eval_site_ids[i]
    actual_ts = eval_series[i]

    actual_slice = actual_ts.slice(pred_ts.start_time(), pred_ts.end_time())

    pred_vals   = pred_ts.values().flatten()
    actual_vals = actual_slice.values().flatten()

    min_len = min(len(pred_vals), len(actual_vals))
    pred_scaled_all.extend(pred_vals[:min_len].tolist())
    actual_scaled_all.extend(actual_vals[:min_len].tolist())
    test_site_ids.extend([site] * min_len)

pred_scaled   = np.array(pred_scaled_all,   dtype="float32")
actual_scaled = np.array(actual_scaled_all, dtype="float32")
test_site_ids = np.array(test_site_ids)

# Inverse-transform to CFS
pred_cfs = pcr.target_scaler.inverse_transform(
    torch.tensor(pred_scaled).reshape(-1, 1)
).numpy().flatten()

actual_cfs = pcr.target_scaler.inverse_transform(
    torch.tensor(actual_scaled).reshape(-1, 1)
).numpy().flatten()

mae_cfs  = float(np.abs(pred_cfs - actual_cfs).mean())
rmse_cfs = float(np.sqrt(np.mean((pred_cfs - actual_cfs) ** 2)))
nse      = float(1 - (
    np.sum((actual_cfs - pred_cfs) ** 2) /
    np.sum((actual_cfs - actual_cfs.mean()) ** 2)
))
bias_pct = float((pred_cfs.mean() - actual_cfs.mean()) / actual_cfs.mean() * 100)

print(f"{'Metric':<12} {'Value':>12}")
print("-" * 26)
print(f"{'MAE':<12} {mae_cfs:>10.1f} CFS")
print(f"{'RMSE':<12} {rmse_cfs:>10.1f} CFS")
print(f"{'NSE':<12} {nse:>12.4f}")
print(f"{'Bias':<12} {bias_pct:>10.1f} %")

wandb.log({
    "test/mae_cfs":  mae_cfs,
    "test/rmse_cfs": rmse_cfs,
    "test/nse":      nse,
    "test/bias_pct": bias_pct,
})

In [ ]:
# ── Per-site metrics table ────────────────────────────────────────────────────
import polars as pl

site_rows = []
for site in np.unique(test_site_ids):
    mask = test_site_ids == site
    a = actual_cfs[mask]
    p = pred_cfs[mask]
    site_mae  = float(np.abs(p - a).mean())
    site_rmse = float(np.sqrt(np.mean((p - a) ** 2)))
    site_nse  = float(
        1 - (np.sum((a - p) ** 2) / np.sum((a - a.mean()) ** 2))
        if a.std() > 0 else float("nan")
    )
    site_rows.append({
        "site_id":            site,
        "actual_mean_cfs":    round(float(a.mean()), 1),
        "predicted_mean_cfs": round(float(p.mean()), 1),
        "mae_cfs":            round(site_mae, 1),
        "rmse_cfs":           round(site_rmse, 1),
        "nse":                round(site_nse, 4),
    })
    print(
        f"Site {site}: actual={a.mean():.1f}  pred={p.mean():.1f}  "
        f"MAE={site_mae:.1f}  NSE={site_nse:.3f}"
    )

site_df = pl.DataFrame(site_rows)
wandb.log({
    "test/per_site_metrics": wandb.Table(
        columns=site_df.columns,
        data=site_df.to_pandas().values.tolist(),
    )
})

In [ ]:
# ── Prediction vs actual time-series chart ────────────────────────────────────
n_plot = 500
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(actual_cfs[:n_plot], label="Actual", alpha=0.8)
ax.plot(pred_cfs[:n_plot],   label="Predicted", alpha=0.8)
ax.set_xlabel("Time step")
ax.set_ylabel("Streamflow (CFS)")
ax.set_title(f"TFT (Darts): Predicted vs Actual (first {n_plot} test samples)")
ax.legend()
plt.tight_layout()
wandb.log({"charts/predictions_vs_actual": wandb.Image(fig)})
plt.show()

In [ ]:
# ── Scatter: predicted vs actual ──────────────────────────────────────────────
sample_idx = np.random.choice(len(actual_cfs), size=min(5000, len(actual_cfs)), replace=False)
scatter_table = wandb.Table(
    columns=["actual_cfs", "predicted_cfs"],
    data=[[float(actual_cfs[i]), float(pred_cfs[i])] for i in sample_idx],
)
wandb.log({
    "charts/scatter_pred_vs_actual": wandb.plot.scatter(
        scatter_table, x="actual_cfs", y="predicted_cfs",
        title="Predicted vs Actual Streamflow (CFS)",
    )
})

In [ ]:
# ── Explainability ────────────────────────────────────────────────────────────
# TFTExplainer surfaces two key interpretability outputs:
#   - Variable selection weights: which features the model attends to most
#   - Attention weights: where in the input window the model focuses
#
# background_series: a representative sample used as the baseline.
# foreground_series: the series to explain (first test site here).
# Both need enough steps: input_chunk_length + output_chunk_length.

import logging
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

_min_len = cfg.input_chunk_length + cfg.output_chunk_length

explainer = TFTExplainer(
    tft,
    background_series=train_series[0][-_min_len * 2:],
    background_past_covariates=train_past_covs[0][-_min_len * 2:],
)

explanation = explainer.explain(
    foreground_series=test_series[0][-_min_len * 2:],
    foreground_past_covariates=test_past_covs[0][-_min_len * 2:],
)

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

# Variable selection weights
fig_var = explainer.plot_variable_selection(explanation)
plt.suptitle("TFT Variable Selection Weights", y=1.02)
plt.tight_layout()
wandb.log({"explainability/variable_selection": wandb.Image(fig_var)})
plt.show()

# Attention weights over the input window
fig_attn = explainer.plot_attention(explanation, plot_type="all")
plt.suptitle("TFT Attention Weights", y=1.02)
plt.tight_layout()
wandb.log({"explainability/attention": wandb.Image(fig_attn)})
plt.show()

In [ ]:
# ── Save model and log as W&B artifact ────────────────────────────────────────
import os
save_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "tft_darts_model.pt")

# Temporarily clear the trainer so Darts skips the PL checkpoint step,
# which fails due to an fsspec version incompatibility.
_trainer = tft.trainer
tft.trainer = None
tft.save(save_path)
tft.trainer = _trainer

print(f"Model saved to {save_path}")

artifact = wandb.Artifact(
    name="tft-darts-model",
    type="model",
    description="Darts TFTModel trained on top-30 flood sites",
    metadata={
        "mae_cfs":  mae_cfs,
        "rmse_cfs": rmse_cfs,
        "nse":      nse,
        **dict(cfg),
    },
)
artifact.add_file(save_path)
run.log_artifact(artifact)
print("Model artifact logged to W&B.")

In [ ]:
wandb.finish()